Check time alignment between eeg and et data by looking at blinks and artifacts

In [1]:
import os
import sys
import mne
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.append('../..')
from data.data_configs import *
from data.data_utils import transform_eeg_time_to_ms, transform_eeg_index_to_ms

### Check difference of max timestamp between EEG and ET
If they are all in a similar range that shows that the time conversion and alignment roughly worked

In [ ]:
for subject in ALL_SUBJECTS:
    events_df = pd.read_csv(os.path.join(LABELED_ET_ALL_DATA_DIR,f"{subject}_labeled_events.csv"),  dtype={'stimulus': str})

    raw_eeg_path = os.path.join(RAW_EEG_DIR, f"{subject}.edf")
    raw = mne.io.read_raw_edf(raw_eeg_path, preload=True, verbose=False)  # preload=True loads data into memory

    eeg_times_ms = transform_eeg_time_to_ms(raw.times)
    et_times_ms = np.asarray(events_df["eeg_time_start"].to_list())

    end_diff = abs(et_times_ms.max() - eeg_times_ms.max())

    print(f"End diff for subject {subject}: {end_diff:.0f} ms")

End diff for subject P01: 9981 ms
End diff for subject P02: 2520 ms
End diff for subject P04: 5519 ms
End diff for subject P05: 3679 ms
End diff for subject P06: 783 ms
End diff for subject P07: 7314 ms
End diff for subject P09: 3138 ms
End diff for subject P10: 599 ms
End diff for subject P11: 825011 ms
End diff for subject P12: 2938 ms
End diff for subject P14: 11863 ms
End diff for subject P15: 595718 ms
End diff for subject P16: 99 ms
End diff for subject P17: 2191 ms
End diff for subject P19: 2465 ms
End diff for subject P20: 6158 ms
End diff for subject P21: 2536 ms
End diff for subject P23: 10494 ms
End diff for subject P24: 3709 ms
End diff for subject P25: 7421 ms
End diff for subject P26: 9599 ms
End diff for subject P27: 3653 ms
End diff for subject P28: 3433 ms
End diff for subject P29: 368346 ms
End diff for subject P30: 2689 ms
End diff for subject P31: 1431 ms
End diff for subject P32: 988992 ms
End diff for subject P33: 12539 ms
End diff for subject P34: 779 ms
End diff

## Check time alignement parameter

For every subject I plot the EEG wave of the Frontal channels which best capture blinks. I overlay the blinks recorded by the eyetracker to see if they align with the peaks. This has to be done manually... I tried using blink detection algorithms to check automatically but that was reliable enough ...

Subjects where the et and eeg data cannot be aligned (meaning the b parameter by the authors is wrong) are removed from the data set

In [17]:
def plot_frontal_eeg_with_blinks(raw, et_blink_times_ms, subject_id=None):

    # Bandpass filter: remove very low (<0.5 Hz) and very high (>30 Hz) frequencies
    raw_filtered = raw.copy().filter(l_freq=0.5, h_freq=30., fir_design='firwin', verbose=False)
    
    # Convert ET blink times to seconds
    et_blink_times_sec = np.array(et_blink_times_ms) / 1000.0  # ms -> s
    
    # Convert ET blink times to EEG sample indices
    sfreq = raw_filtered.info['sfreq']
    et_blink_samples = (et_blink_times_sec * sfreq).astype(int)
    
    # Create MNE events array
    events = np.zeros((len(et_blink_samples), 3), dtype=int)
    events[:, 0] = et_blink_samples
    events[:, 2] = 1  # blink event ID
    
    # Create annotations for visualization
    annotations = mne.Annotations(
        onset=et_blink_times_sec,
        duration=np.zeros_like(et_blink_times_sec),
        description=['blink']*len(et_blink_times_sec)
    )
    raw_filtered.set_annotations(annotations)
    
    # Print sanity check
    print(f"Subject {subject_id} EEG start/end (s): {raw_filtered.times[0]:.2f} / {raw_filtered.times[-1]:.2f}")
    print(f"Subject {subject_id} ET blink start/end (s): {et_blink_times_sec.min():.2f} / {et_blink_times_sec.max():.2f}")


    
    # Pick frontal channels
    frontal_raw = raw_filtered.copy().pick(ARTIFACT_CHANNELS)
    
    # Enable Qt backend for scrollable plot
    %matplotlib qt
    
    # Plot and block until closed
    title = f"Subject {subject_id} - Frontal EEG with ET blinks" if subject_id else "Frontal EEG with ET blinks"
    frontal_raw.plot(
        scalings={'eeg': 100e-6},
        show=True,
        block=True,                # block execution until window is closed
        title=title,
        events=events,
        event_color={1: 'green'},
        verbose=False
    )


In [18]:
for subject in ALL_SUBJECTS[0:1]:
    events_df = pd.read_csv(os.path.join(LABELED_ET_ALL_DATA_DIR,f"{subject}_labeled_events.csv"),  dtype={'stimulus': str})

    blinks_df = events_df[events_df["event_type"] == "Blink"]
    blinks_df = blinks_df.sort_values(by='start_time')
    et_blinks = blinks_df["eeg_time_start"].to_list()

    raw_eeg_path = os.path.join(RAW_EEG_DIR, f"{subject}.edf")
    raw_eeg = mne.io.read_raw_edf(raw_eeg_path, preload=True, verbose=False)

    plot_frontal_eeg_with_blinks(raw_eeg, et_blinks, subject_id=subject)
    break


Subject P01 EEG start/end (s): 0.00 / 2174.00
Subject P01 ET blink start/end (s): 266.13 / 2160.35


### Make sure that raw and processed data are the same...

In [ ]:
for subject in ALL_SUBJECTS:

    raw_eeg_path = os.path.join(RAW_EEG_DIR, f"{subject}.edf")
    raw = mne.io.read_raw_edf(raw_eeg_path, preload=False, verbose=False)  # preload=True loads data into memory

    clean_eeg_path = os.path.join(EEG_DIR, f"{subject}.edf")
    clean = mne.io.read_raw_edf(clean_eeg_path, preload=False, verbose=False)  # preload=True loads data into memory

    eeg_times_ms = transform_eeg_time_to_ms(raw.times)
    clean_times_ms = transform_eeg_time_to_ms(clean.times)

    assert np.array_equal(eeg_times_ms, clean_times_ms)

### Look at target and non-target fixations

In [30]:
def plot_frontal_eeg_with_fixations(raw, target_fixations_ms, nontarget_fixations_ms, subject_id=None):

    # Bandpass filter: remove very low (<0.5 Hz) and very high (>30 Hz) frequencies
    raw_filtered = raw.copy().filter(l_freq=0.5, h_freq=30., fir_design='firwin', verbose=False)
    
    # Convert fixation times to seconds
    target_fixations_sec = np.array(target_fixations_ms) / 1000.0  # ms -> s
    nontarget_fixations_sec = np.array(nontarget_fixations_ms) / 1000.0  # ms -> s
    
    # Convert fixation times to EEG sample indices
    sfreq = raw_filtered.info['sfreq']
    target_samples = (target_fixations_sec * sfreq).astype(int)
    nontarget_samples = (nontarget_fixations_sec * sfreq).astype(int)
    
    # Create MNE events array for target fixations
    events_target = np.zeros((len(target_samples), 3), dtype=int)
    events_target[:, 0] = target_samples
    events_target[:, 2] = 1  # target fixation event ID
    
    # Create MNE events array for non-target fixations
    events_nontarget = np.zeros((len(nontarget_samples), 3), dtype=int)
    events_nontarget[:, 0] = nontarget_samples
    events_nontarget[:, 2] = 2  # non-target fixation event ID
    
    # Combine events
    if len(events_target) > 0 and len(events_nontarget) > 0:
        events = np.vstack([events_target, events_nontarget])
    elif len(events_target) > 0:
        events = events_target
    else:
        events = events_nontarget
    
    # Create annotations for visualization
    all_fixations_sec = np.concatenate([target_fixations_sec, nontarget_fixations_sec])
    all_descriptions = (['target'] * len(target_fixations_sec) + 
                       ['nontarget'] * len(nontarget_fixations_sec))
    
    annotations = mne.Annotations(
        onset=all_fixations_sec,
        duration=np.zeros_like(all_fixations_sec),
        description=all_descriptions
    )
    raw_filtered.set_annotations(annotations)
    
    # Print sanity check
    print(f"Subject {subject_id} EEG start/end (s): {raw_filtered.times[0]:.2f} / {raw_filtered.times[-1]:.2f}")
    print(f"Subject {subject_id} Target fixations: {len(target_fixations_ms)}")
    print(f"Subject {subject_id} Non-target fixations: {len(nontarget_fixations_ms)}")
    if len(target_fixations_sec) > 0:
        print(f"Subject {subject_id} Target fix start/end (s): {target_fixations_sec.min():.2f} / {target_fixations_sec.max():.2f}")
    if len(nontarget_fixations_sec) > 0:
        print(f"Subject {subject_id} Non-target fix start/end (s): {nontarget_fixations_sec.min():.2f} / {nontarget_fixations_sec.max():.2f}")
    
    # Pick frontal channels
    frontal_raw = raw_filtered.copy().pick(DEFAULT_EEGNET_CHANNELS)
    
    # Enable Qt backend for scrollable plot
    %matplotlib qt
    
    # Plot with much shorter duration for closer view
    title = f"Subject {subject_id} - Frontal EEG with Target/Non-Target Fixations" if subject_id else "Frontal EEG with Target/Non-Target Fixations"
    frontal_raw.plot(
        scalings={'eeg': 20e-6},   # Smaller scaling for closer view
        show=True,
        block=True,                # block execution until window is closed
        title=title,
        events=events,
        event_color={1: 'red', 2: 'blue'},  # red=target, blue=non-target
        duration=3,               # Show only 3 seconds at a time for closer view
        start=0,                   # Start at beginning
        time_format='float',       # Show time with decimals
        verbose=False
    )

In [ ]:
# Plot frontal EEG with target/non-target fixations for P01
for subject in ALL_SUBJECTS[:1]:  # Start with P01
    events_df = pd.read_csv(os.path.join(LABELED_ET_ALL_DATA_DIR, f"{subject}_labeled_events.csv"), dtype={'stimulus': str})

    fixations_df = events_df[events_df["event_type"] == "Fixation"]
    fixations_df = fixations_df.sort_values(by='start_time')
    
    # Separate target and non-target fixations
    target_fixations = fixations_df[fixations_df["fixation_on_airplane"] == True]["eeg_time_start"].to_list()
    nontarget_fixations = fixations_df[fixations_df["fixation_on_airplane"] == False]["eeg_time_start"].to_list()

    raw_eeg_path = os.path.join(EEG_DIR, f"{subject}.edf")
    raw_eeg = mne.io.read_raw_edf(raw_eeg_path, preload=True, verbose=False)

    plot_frontal_eeg_with_fixations(raw_eeg, target_fixations, nontarget_fixations, subject_id=subject)
    break

Subject P01 EEG start/end (s): 0.00 / 2174.00
Subject P01 Target fixations: 2370
Subject P01 Non-target fixations: 2059
Subject P01 Target fix start/end (s): 248.10 / 2163.80
Subject P01 Non-target fix start/end (s): 247.88 / 2164.02


: 